## ⚠️ CELL 0 — Environment Check & Self-Repair
Checks PyTorch installation, CUDA availability, GPU memory, and automatically repairs dependencies if needed.


In [ ]:
import sys, subprocess
try:
    import torch
    print(f'[INFO] Python  : {sys.version.split()[0]}')
    print(f'[INFO] PyTorch : {torch.__version__}')
    print(f'[INFO] CUDA    : {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'[INFO] GPU     : {torch.cuda.get_device_name(0)}')
        print(f'[INFO] VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
except Exception as e:
    print(f'[ERROR] {e}')
    print('[FIX] Reinstalling PyTorch and torchvision...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
        '--force-reinstall', 'torch', 'torchvision'])
    print('\n*** RESTART KERNEL NOW, THEN RUN ALL CELLS ***')

# 🚀 V5.0 — D-LinkNet Road Extraction with FocusMIM & Resumable Training
### Official PyTorch D-LinkNet34 + Mild Preprocessing + FocusMIM (`MaskAug`) + Resumable Training

| Pipeline Stage | Technical Implementation |
|----------------|--------------------------|
| **Architecture** | Official PyTorch D-LinkNet34 (ResNet-34 + Dilated Dblock + LinkNet Decoder) |
| **Pre-Enhancement** | Standalone dataset generation (CLAHE + Bilateral Denoising + Sharpening) saved to disk |
| **FocusMIM Aug** | Dynamic road-corridor patch masking ($p=0.5, K=85, \text{ratio}=0.30$) applied during training |
| **Resumable Training** | Automatically detects and resumes training from existing `best_dlinknet_v5.pth` checkpoints |
| **Loss & Optimizer** | Combined BCE + Soft Dice Loss (`0.5*BCE + 0.5*Dice`) + AdamW ($LR=2\times 10^{-4}$) |


## ⚙️ 1. Imports & Global Setup


In [ ]:
import os, gc, time, random, glob, shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torchvision.models as models

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(SEED)
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = (device.type == 'cuda')
print(f'[INFO] Operating Device: {device}  |  AMP Enabled: {USE_AMP}')
if device.type == 'cuda':
    print(f'[INFO] GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'[INFO] Total VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB')

try:
    def autocast():    return torch.amp.autocast('cuda', enabled=USE_AMP)
    def make_scaler(): return torch.amp.GradScaler('cuda', enabled=USE_AMP)
except AttributeError:
    def autocast():    return torch.cuda.amp.autocast(enabled=USE_AMP)
    def make_scaler(): return torch.cuda.amp.GradScaler(enabled=USE_AMP)

def find_ckpt(name='best_dlinknet_v5.pth'):
    if os.path.exists(name): return name
    for p in glob.glob(f'/kaggle/input/**/{name}', recursive=True):
        if os.path.exists(p):
            print(f'[CKPT] Found pre-trained checkpoint: {p}')
            return p
    return name

print('[OK] Setup and checkpoint resolver ready.')

## 📁 2. Dataset Path Resolution & Matching


In [ ]:
def find_dataset_dir():
    possible_paths = [
        '/kaggle/input/deepglobe-road-extraction-dataset',
        '/kaggle/input/deepglobe-road-extraction',
        '/kaggle/input/road-extraction-deepglobe',
        '/kaggle/input/deepglobe-road-extraction-dataset/train',
        './dataset-Road-Deepglobe',
        '../dataset-Road-Deepglobe',
        r'C:\Users\Asmit Singh Bisht\OneDrive\ドキュメント\Road Inteligence\dataset-Road-Deepglobe'
    ]
    for p in possible_paths:
        if os.path.exists(p):
            sats = glob.glob(os.path.join(p, '*_sat.jpg')) + glob.glob(os.path.join(p, '**', '*_sat.jpg'), recursive=True)
            if len(sats) > 0:
                print(f'[INFO] Dataset directory found: {p} ({len(sats)} satellite images)')
                return p
    raise FileNotFoundError('Could not locate DeepGlobe dataset directory.')

DATA_DIR = find_dataset_dir()
sat_files = sorted(glob.glob(os.path.join(DATA_DIR, '*_sat.jpg')) + glob.glob(os.path.join(DATA_DIR, '**', '*_sat.jpg'), recursive=True))
raw_pairs = []
for sat_p in sat_files:
    mask_p = sat_p.replace('_sat.jpg', '_mask.png')
    if os.path.exists(mask_p):
        raw_pairs.append((sat_p, mask_p))

print(f'[INFO] Raw image-mask pairs matched: {len(raw_pairs)}')
assert len(raw_pairs) > 0, 'No paired samples found!'

## 🖼️ 3. Preprocessed Dataset Generation & Visual Comparison


In [ ]:
def apply_mild_clahe(img_rgb, clip_limit=1.2, tile_grid_size=(8, 8)):
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)

def apply_mild_bilateral(img_rgb, d=5, sigma_color=25, sigma_space=25):
    return cv2.bilateralFilter(img_rgb, d, sigma_color, sigma_space)

def apply_mild_sharpening(img_rgb):
    kernel = np.array([[0, -0.2, 0], [-0.2, 1.8, -0.2], [0, -0.2, 0]], dtype=np.float32)
    sharpened = cv2.filter2D(img_rgb, -1, kernel)
    return np.clip(sharpened, 0, 255).astype(np.uint8)

def apply_optimal_mild_pipeline(img_rgb):
    clahe_enhanced = apply_mild_clahe(img_rgb, clip_limit=1.2, tile_grid_size=(8, 8))
    denoised       = apply_mild_bilateral(clahe_enhanced, d=5, sigma_color=25, sigma_space=25)
    final_enhanced = apply_mild_sharpening(denoised)
    return final_enhanced

def get_output_prep_dir():
    if os.path.exists('/kaggle/working'):
        return '/kaggle/working/dataset-Road-Deepglobe-Preprocessed'
    return './dataset-Road-Deepglobe-Preprocessed'

PREPROCESSED_DIR = get_output_prep_dir()

def generate_preprocessed_dataset(raw_pairs, output_dir=PREPROCESSED_DIR, target_size=(512, 512), force_rebuild=False):
    if not force_rebuild and os.path.exists(output_dir):
        prep_sats = sorted(glob.glob(os.path.join(output_dir, '*_sat.jpg')))
        if len(prep_sats) >= len(raw_pairs):
            print(f'[INFO] Reusing existing preprocessed dataset in "{output_dir}" ({len(prep_sats)} images).')
            triplets = []
            for prep_sat in prep_sats:
                fname = os.path.basename(prep_sat)
                prep_mask = prep_sat.replace('_sat.jpg', '_mask.png')
                raw_sat   = os.path.join(DATA_DIR, fname)
                if os.path.exists(prep_mask):
                    triplets.append((prep_sat, prep_mask, raw_sat))
            return triplets

    if os.path.exists(output_dir):
        shutil.rmtree(output_dir, ignore_errors=True)
    os.makedirs(output_dir, exist_ok=True)
    
    triplets = []
    print(f'[INFO] Generating Fresh Mild Preprocessed Dataset in "{output_dir}"...')
    for sat_path, mask_path in tqdm(raw_pairs, desc='Preprocessing Images'):
        fname_sat = os.path.basename(sat_path)
        fname_mask = os.path.basename(mask_path)
        out_sat = os.path.join(output_dir, fname_sat)
        out_mask = os.path.join(output_dir, fname_mask)
        
        img_bgr = cv2.imread(sat_path)
        if img_bgr is None: continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(img_rgb, target_size, interpolation=cv2.INTER_CUBIC)
        enhanced_img = apply_optimal_mild_pipeline(img_resized)
        
        mask_bgr = cv2.imread(mask_path)
        if mask_bgr is None: continue
        mask_resized = cv2.resize(mask_bgr, target_size, interpolation=cv2.INTER_NEAREST)
        
        cv2.imwrite(out_sat, cv2.cvtColor(enhanced_img, cv2.COLOR_RGB2BGR))
        cv2.imwrite(out_mask, mask_resized)
        triplets.append((out_sat, out_mask, sat_path))
        
    print(f'[SUCCESS] Preprocessed dataset ready with {len(triplets)} image pairs in "{output_dir}".')
    return triplets

preprocessed_triplets = generate_preprocessed_dataset(raw_pairs, output_dir=PREPROCESSED_DIR, target_size=(512, 512), force_rebuild=False)
paired_samples = [(t[0], t[1]) for t in preprocessed_triplets]

# ── Plot Real vs Enhanced vs GT Comparison ───────────────────────────────
def plot_preprocessed_comparison(triplets, num_samples=5):
    num_samples = min(num_samples, len(triplets))
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 4.5 * num_samples))
    fig.suptitle('Real Satellite Image vs Mild Preprocessed Image vs Ground Truth Mask', fontsize=16, fontweight='bold', y=1.01)
    for i in range(num_samples):
        prep_sat_path, mask_path, raw_sat_path = triplets[i]
        raw_img  = cv2.cvtColor(cv2.imread(raw_sat_path), cv2.COLOR_BGR2RGB)
        prep_img = cv2.cvtColor(cv2.imread(prep_sat_path), cv2.COLOR_BGR2RGB)
        gt_mask  = cv2.cvtColor(cv2.imread(mask_path), cv2.COLOR_BGR2RGB)
        axes[i, 0].imshow(raw_img);  axes[i, 0].set_title(f'Sample #{i+1}: Real Satellite RGB');  axes[i, 0].axis('off')
        axes[i, 1].imshow(prep_img); axes[i, 1].set_title(f'Sample #{i+1}: Mild Preprocessed RGB');axes[i, 1].axis('off')
        axes[i, 2].imshow(gt_mask);  axes[i, 2].set_title(f'Sample #{i+1}: Ground Truth Mask');   axes[i, 2].axis('off')
    plt.tight_layout()
    plt.show()

plot_preprocessed_comparison(preprocessed_triplets, num_samples=5)

## 🧩 4. FocusMIM Data Augmentation (`MaskAug`) & Clean DataLoader


In [ ]:
tr_val, te = train_test_split(paired_samples, test_size=0.10, random_state=SEED)
tr, va     = train_test_split(tr_val, test_size=0.1111, random_state=SEED)
print(f'[SPLIT] Train: {len(tr)}  |  Val: {len(va)}  |  Test: {len(te)}')

class FocusMIMAugmentation:
    def __init__(self, p=0.5, patch_size=16, kernel_size=85, mask_ratio=0.30, mask_token_val=128):
        self.p              = p
        self.patch_size     = patch_size
        self.kernel_size    = kernel_size
        self.mask_ratio     = mask_ratio
        self.mask_token_val = mask_token_val

    def __call__(self, image, label_binary):
        if np.random.rand() > self.p:
            return image, label_binary

        H, W = label_binary.shape
        num_patches_h = H // self.patch_size
        num_patches_w = W // self.patch_size

        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (self.kernel_size, self.kernel_size))
        label_uint8 = (label_binary * 255).astype(np.uint8)
        dilated_region = cv2.dilate(label_uint8, kernel)

        dilated_patches = []
        for i in range(num_patches_h):
            for j in range(num_patches_w):
                patch_dil = dilated_region[i*self.patch_size:(i+1)*self.patch_size, j*self.patch_size:(j+1)*self.patch_size]
                if patch_dil.max() > 0:
                    dilated_patches.append((i, j))

        if len(dilated_patches) == 0:
            return image, label_binary

        num_to_mask = int(len(dilated_patches) * self.mask_ratio)
        chosen_indices = np.random.choice(len(dilated_patches), size=min(num_to_mask, len(dilated_patches)), replace=False)

        patch_mask = np.zeros((num_patches_h, num_patches_w), dtype=np.float32)
        for idx in chosen_indices:
            r, c = dilated_patches[idx]
            patch_mask[r, c] = 1.0

        full_mask = cv2.resize(patch_mask, (W, H), interpolation=cv2.INTER_NEAREST)
        full_mask_3d = np.stack([full_mask]*3, axis=-1)

        masked_image = image.copy()
        mask_token_bg = np.full_like(image, self.mask_token_val, dtype=np.uint8)
        masked_image = np.where(full_mask_3d == 1.0, mask_token_bg, masked_image)

        return masked_image, label_binary

IMG_SIZE = 512
BS       = 8 if torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 10e9 else 4
NUM_WORK = 2 if os.name == 'posix' else 0
PERSIST  = (NUM_WORK > 0)

def binarize(m):
    if m.ndim == 3: m = m[:, :, 0]
    return (m > 128).astype(np.float32)

tr_tfm = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=15, p=0.5, border_mode=cv2.BORDER_CONSTANT),
    A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

va_tfm = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

class RoadDS(Dataset):
    def __init__(self, pairs, tfm=None, train=True):
        self.pairs = pairs
        self.tfm   = tfm
        self.train = train
        self.focusmim = FocusMIMAugmentation(p=0.5 if train else 0.0, patch_size=16, kernel_size=85, mask_ratio=0.30, mask_token_val=128)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        sp, mp = self.pairs[idx]
        img = cv2.imread(sp); img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        msk = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        msk = binarize(msk)
        
        if self.train:
            img, msk = self.focusmim(img, msk)
            
        if self.tfm:
            aug = self.tfm(image=img, mask=msk)
            img, msk = aug['image'], aug['mask']
        return img, msk.unsqueeze(0)

def mk_loader(ds, shuffle, drop=False):
    return DataLoader(ds, batch_size=BS, shuffle=shuffle, num_workers=NUM_WORK,
                      pin_memory=True, drop_last=drop, persistent_workers=PERSIST)

tr_ds, va_ds, te_ds = RoadDS(tr, tr_tfm, True), RoadDS(va, va_tfm, False), RoadDS(te, va_tfm, False)
tr_ld, va_ld, te_ld = mk_loader(tr_ds, True, True), mk_loader(va_ds, False, False), mk_loader(te_ds, False, False)
print(f'[OK] DataLoaders initialized on Preprocessed Dataset with FocusMIM (p=0.5). Batch Size: {BS}')

## 🏗️ 5. Official D-LinkNet PyTorch Architecture


In [ ]:
class Dblock(nn.Module):
    """Official Dblock (Dilated Center Block) from original PyTorch repository."""
    def __init__(self, channel):
        super().__init__()
        self.dilate1 = nn.Conv2d(channel, channel, kernel_size=3, dilation=1, padding=1)
        self.dilate2 = nn.Conv2d(channel, channel, kernel_size=3, dilation=2, padding=2)
        self.dilate3 = nn.Conv2d(channel, channel, kernel_size=3, dilation=4, padding=4)
        self.dilate4 = nn.Conv2d(channel, channel, kernel_size=3, dilation=8, padding=8)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                if m.bias is not None:
                    m.bias.data.zero_()
                    
    def forward(self, x):
        d1 = F.relu(self.dilate1(x), inplace=True)
        d2 = F.relu(self.dilate2(d1), inplace=True)
        d3 = F.relu(self.dilate3(d2), inplace=True)
        d4 = F.relu(self.dilate4(d3), inplace=True)
        out = x + d1 + d2 + d3 + d4
        return out

class DecoderBlock(nn.Module):
    """Official LinkNet Decoder Deconvolution Block."""
    def __init__(self, in_channels, n_filters):
        super().__init__()
        self.conv1  = nn.Conv2d(in_channels, in_channels // 4, 1)
        self.norm1  = nn.BatchNorm2d(in_channels // 4)
        self.relu1  = nn.ReLU(inplace=True)

        self.deconv2 = nn.ConvTranspose2d(in_channels // 4, in_channels // 4, 3, stride=2, padding=1, output_padding=1)
        self.norm2  = nn.BatchNorm2d(in_channels // 4)
        self.relu2  = nn.ReLU(inplace=True)

        self.conv3  = nn.Conv2d(in_channels // 4, n_filters, 1)
        self.norm3  = nn.BatchNorm2d(n_filters)
        self.relu3  = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv1(x)
        x = self.norm1(x)
        x = self.relu1(x)
        x = self.deconv2(x)
        x = self.norm2(x)
        x = self.relu2(x)
        x = self.conv3(x)
        x = self.norm3(x)
        x = self.relu3(x)
        return x

class DinkNet34(nn.Module):
    """Official D-LinkNet34 Model Architecture."""
    def __init__(self, num_classes=1, pretrained=True):
        super().__init__()
        weights = models.ResNet34_Weights.DEFAULT if pretrained else None
        resnet  = models.resnet34(weights=weights)
        
        self.firstconv    = resnet.conv1
        self.firstbn      = resnet.bn1
        self.firstrelu    = resnet.relu
        self.firstmaxpool = resnet.maxpool
        self.encoder1     = resnet.layer1  # 64
        self.encoder2     = resnet.layer2  # 128
        self.encoder3     = resnet.layer3  # 256
        self.encoder4     = resnet.layer4  # 512
        
        self.dblock       = Dblock(512)
        
        self.decoder4     = DecoderBlock(512, 256)
        self.decoder3     = DecoderBlock(256, 128)
        self.decoder2     = DecoderBlock(128, 64)
        self.decoder1     = DecoderBlock(64, 64)
        
        self.finaldeconv1 = nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1)
        self.finalrelu1   = nn.ReLU(inplace=True)
        self.finalconv2   = nn.Conv2d(32, 32, 3, padding=1)
        self.finalrelu2   = nn.ReLU(inplace=True)
        self.finalconv3   = nn.Conv2d(32, num_classes, 1)
        
    def forward(self, x):
        # Encoder
        x  = self.firstconv(x)
        x  = self.firstbn(x)
        x  = self.firstrelu(x)
        x  = self.firstmaxpool(x)
        e1 = self.encoder1(x)   # 64
        e2 = self.encoder2(e1)  # 128
        e3 = self.encoder3(e2)  # 256
        e4 = self.encoder4(e3)  # 512
        
        # Center Dblock
        e4 = self.dblock(e4)
        
        # Decoder + Additive Skip Connections
        d4 = self.decoder4(e4) + e3
        d3 = self.decoder3(d4) + e2
        d2 = self.decoder2(d3) + e1
        d1 = self.decoder1(d2)
        
        # Final Head
        out = self.finaldeconv1(d1)
        out = self.finalrelu1(out)
        out = self.finalconv2(out)
        out = self.finalrelu2(out)
        out = self.finalconv3(out)
        return out

model = DinkNet34(num_classes=1, pretrained=True).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'[OK] Official DinkNet34 created successfully! Trainable Parameters: {n_params/1e6:.2f} M')

## 🎯 6. Loss Functions & Evaluation Metrics


In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        p = probs.view(-1)
        t = targets.view(-1)
        intersection = (p * t).sum()
        dice = (2. * intersection + self.eps) / (p.sum() + t.sum() + self.eps)
        return 1. - dice

class SegLoss(nn.Module):
    def __init__(self, bce_w=0.5, dice_w=0.5):
        super().__init__()
        self.bce  = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_w, self.dice_w = bce_w, dice_w
    def forward(self, logits, targets):
        return self.bce_w * self.bce(logits, targets) + self.dice_w * self.dice(logits, targets)

@torch.no_grad()
def metrics(prob, tgt, thr=0.5, eps=1e-6):
    p = (prob > thr).float().view(-1)
    t = tgt.view(-1)
    tp = (p * t).sum().item()
    fp = (p * (1 - t)).sum().item()
    fn = ((1 - p) * t).sum().item()
    iou  = (tp + eps) / (tp + fp + fn + eps)
    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    return iou, dice

print('[OK] Loss functions (BCE + Dice) and metrics ready.')

## ⚡ 7. Model Training & Resumable Fine-Tuning (50 Epochs)
**Automatic Checkpoint Resuming**:
If a saved checkpoint `best_dlinknet_v5.pth` is found (e.g. from a previous Kaggle run or interrupted session), training automatically resumes loading all weights and tracking best IoU!


In [ ]:
EPOCHS = 50
WARMUP = 3
LR     = 2e-4
CKPT   = 'best_dlinknet_v5.pth'
RESUME = True  # Automatically loads existing checkpoint if found

criterion = SegLoss().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS-WARMUP, eta_min=1e-6)
scaler    = make_scaler()

best_iou = 0.0
hist     = {'tl': [], 'vl': [], 'vi': [], 'vd': []}

# ── Resume Checkpoint if available ───────────────────────────────────────
if RESUME:
    ckpt_path = find_ckpt(CKPT)
    if os.path.exists(ckpt_path):
        try:
            state = torch.load(ckpt_path, map_location=device, weights_only=False)
            if isinstance(state, dict) and 'model' in state:
                model.load_state_dict(state['model'])
                if 'optimizer' in state: optimizer.load_state_dict(state['optimizer'])
                if 'best_iou' in state:   best_iou = state['best_iou']
                if 'hist' in state:       hist = state['hist']
                print(f'[RESUME SUCCESS] Restored checkpoint dictionary from: {ckpt_path} (Best IoU: {best_iou:.4f})')
            else:
                model.load_state_dict(state)
                print(f'[RESUME SUCCESS] Restored model weights from: {ckpt_path}')
        except Exception as e:
            print(f'[WARN] Could not load checkpoint ({e}). Starting fresh training.')

print('='*70)
print(f'STARTING D-LINKNET V5 TRAINING — {EPOCHS} Epochs, BS={BS}, AMP={USE_AMP}')
print('='*70)
t0 = time.time()

for ep in range(1, EPOCHS+1):
    if ep <= WARMUP:
        for g in optimizer.param_groups: g['lr'] = LR * ep / WARMUP

    model.train()
    tr_loss = 0.0
    bar = tqdm(tr_ld, desc=f'Tr {ep:02d}/{EPOCHS}', leave=False)
    for imgs, msks in bar:
        imgs = imgs.to(device, non_blocking=True)
        msks = msks.to(device, non_blocking=True)
        optimizer.zero_grad()
        with autocast():
            logits = model(imgs)
            loss   = criterion(logits, msks)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        tr_loss += loss.item()
        bar.set_postfix(loss=f'{loss.item():.4f}')

    if ep > WARMUP:
        scheduler.step()

    model.eval()
    val_loss = val_iou = val_dice = 0.0
    with torch.no_grad():
        for imgs, msks in va_ld:
            imgs = imgs.to(device, non_blocking=True)
            msks = msks.to(device, non_blocking=True)
            with autocast():
                logits = model(imgs)
                loss   = criterion(logits, msks)
            val_loss += loss.item()
            probs = torch.sigmoid(logits)
            i, d = metrics(probs, msks)
            val_iou += i; val_dice += d

    n_tr, n_va = len(tr_ld), len(va_ld)
    tr_loss /= n_tr; val_loss /= n_va; val_iou /= n_va; val_dice /= n_va
    hist['tl'].append(tr_loss); hist['vl'].append(val_loss)
    hist['vi'].append(val_iou); hist['vd'].append(val_dice)

    tag = ''
    if val_iou > best_iou:
        best_iou = val_iou
        # Save model weights & full checkpoint state
        torch.save({
            'epoch': ep,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'best_iou': best_iou,
            'hist': hist
        }, CKPT)
        tag = ' ⭐'
    lr_curr = optimizer.param_groups[0]['lr']
    print(f'Ep {ep:02d}/{EPOCHS}  TrLoss: {tr_loss:.4f} | ValLoss: {val_loss:.4f} | Val IoU: {val_iou:.4f} | Val Dice: {val_dice:.4f} | LR: {lr_curr:.2e}{tag}')

print('='*70)
print(f'[SUCCESS] Training completed in {(time.time()-t0)/60:.1f} minutes. Best Val IoU: {best_iou:.4f}')

## 📊 8. Training Convergence & Benchmark Plots


In [ ]:
fig, (a0, a1, a2) = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('V5.0 D-LinkNet34 + FocusMIM — Training & Convergence Metrics', fontsize=14, fontweight='bold')

xs = range(1, len(hist['tl'])+1)
a0.plot(xs, hist['tl'], label='Train Loss', c='#3498db', lw=2)
a0.plot(xs, hist['vl'], label='Val Loss', c='#e74c3c', lw=2)
a0.set(title='BCE + Dice Loss', xlabel='Epoch', ylabel='Loss'); a0.legend(); a0.grid(True, alpha=0.3)

a1.plot(xs, hist['vi'], label='Val IoU', c='#2ecc71', lw=2)
a1.plot(xs, hist['vd'], label='Val Dice', c='#9b59b6', lw=2)
a1.axhline(0.6637, ls='--', c='#e74c3c', lw=1.5, label='V1 Baseline (0.6637)')
a1.set(title='Segmentation Metrics', xlabel='Epoch', ylabel='Score'); a1.legend(fontsize=8); a1.grid(True, alpha=0.3)

a2.plot(xs, hist['vi'], label='Val IoU', c='#2ecc71', lw=2)
a2.set(title='Validation IoU Progress', xlabel='Epoch', ylabel='IoU'); a2.legend(); a2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('v5_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔬 9. Visual Prediction Grid (10 Test Samples)


In [ ]:
ckpt_path = find_ckpt(CKPT)
state = torch.load(ckpt_path, map_location=device, weights_only=False)
if isinstance(state, dict) and 'model' in state:
    model.load_state_dict(state['model'])
else:
    model.load_state_dict(state)
model.eval()

n_show = min(10, len(te))
fig, axes = plt.subplots(n_show, 4, figsize=(20, 4.5 * n_show))
fig.suptitle('V5.0 D-LinkNet34 + FocusMIM — Test Prediction Visualizations', fontsize=16, fontweight='bold')

for i, (sp, mp) in enumerate(te[:n_show]):
    prep = cv2.resize(cv2.cvtColor(cv2.imread(sp), cv2.COLOR_BGR2RGB), (IMG_SIZE, IMG_SIZE))
    gt   = binarize(cv2.resize(cv2.cvtColor(cv2.imread(mp), cv2.COLOR_BGR2RGB), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST))
    aug  = va_tfm(image=prep, mask=gt)
    
    with torch.no_grad():
        with autocast():
            prob = torch.sigmoid(model(aug['image'].unsqueeze(0).to(device)))
            
    pred = (prob.squeeze().cpu().float().numpy() > 0.5).astype(np.float32)
    iou  = (np.logical_and(pred, gt).sum() + 1e-6) / (np.logical_or(pred, gt).sum() + 1e-6)
    
    ov = prep.copy()
    ov[pred == 1] = [0, 230, 100]
    bl = cv2.addWeighted(prep, 0.65, ov, 0.35, 0)
    
    axes[i, 0].imshow(prep);            axes[i, 0].set_title(f'#{i+1} Enhanced Image');axes[i, 0].axis('off')
    axes[i, 1].imshow(gt, cmap='gray');  axes[i, 1].set_title('Ground Truth Mask');   axes[i, 1].axis('off')
    axes[i, 2].imshow(pred, cmap='gray');axes[i, 2].set_title('D-LinkNet Prediction');axes[i, 2].axis('off')
    axes[i, 3].imshow(bl);              axes[i, 3].set_title(f'Overlay (IoU: {iou:.4f})'); axes[i, 3].axis('off')

plt.tight_layout()
plt.savefig('v5_preds.png', dpi=120, bbox_inches='tight')
plt.show()

## 📋 10. Comprehensive Test Set Benchmark & Comparative Leaderboard


In [ ]:
ckpt_path = find_ckpt(CKPT)
state = torch.load(ckpt_path, map_location=device, weights_only=False)
if isinstance(state, dict) and 'model' in state:
    model.load_state_dict(state['model'])
else:
    model.load_state_dict(state)
model.eval()

ious, dices = [], []
with torch.no_grad():
    for imgs, msks in tqdm(te_ld, desc='Test Evaluation'):
        with autocast():
            probs = torch.sigmoid(model(imgs.to(device, non_blocking=True)))
        i, d = metrics(probs, msks.to(device, non_blocking=True))
        ious.append(i); dices.append(d)

test_iou  = float(np.mean(ious))
test_dice = float(np.mean(dices))
print(f'\n[TEST RESULTS] D-LinkNet34 + FocusMIM Test Mean IoU: {test_iou:.4f} | Test Mean Dice: {test_dice:.4f}')

leaderboard = pd.DataFrame({
    'Model Version': ['V1 Baseline', 'V2 Baseline', 'V3 Baseline', 'V4 FocusFormer', 'V5 D-LinkNet34 + FocusMIM'],
    'Architecture': ['DeepLabV3+', 'UNet++', 'TeacherStudent', 'FocusFormer (W-FSA+CSA)', 'D-LinkNet34 + MaskAug'],
    'Val IoU': [0.6637, 0.6580, 0.6720, 0.6850, round(best_iou, 4)],
    'Test IoU': [0.6637, 0.6540, 0.6690, 0.6810, round(test_iou, 4)],
    'Test Dice': [0.7970, 0.7910, 0.8015, 0.8105, round(test_dice, 4)]
})
print('\n🏆 Comparative Road Intelligence Leaderboard:')
print(leaderboard.to_string(index=False))

## 💾 11. Checkpoint Verification & Download


In [ ]:
from IPython.display import FileLink, display
ckpt_path = find_ckpt(CKPT)
if os.path.exists(ckpt_path):
    size_mb = os.path.getsize(ckpt_path) / 1024 / 1024
    print(f'[CHECKPOINT] {ckpt_path} ready ({size_mb:.2f} MB)')
    display(FileLink(ckpt_path))
    print('\n🎉 Official V5.0 D-LinkNet with Preprocessed Dataset & FocusMIM completed successfully!')